[final_test](https://github.com/OpenGVLab/VideoMAEv2/blob/master/engine_for_finetuning.py#L218)

In [8]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

from IPython.display import Video
from IPython.display import Audio as play_audio
from torchvision.utils import make_grid
from torchcodec.encoders import VideoEncoder

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

import matplotlib.pyplot as plt

import sys
sys.path.append('../../../../')

In [9]:
from pathlib import Path
import math
import time
import random
import datetime
from functools import partial
from collections import OrderedDict

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image

%load_ext autoreload
%autoreload 2

from computer_vision.video_mae.finetune_parameter_parser import parser
from computer_vision.video_mae.dataset.datasets import VideoClsDataset

from computer_vision.torch_video.utils.plotting import plot_all
from computer_vision.torch_video.utils.progress import save_metrics
from computer_vision.video_mae.models.modeling_finetune import vit_tiny_patch16_224
from computer_vision.video_mae.utils import multiple_samples_collate, seed_worker, load_state_dict, MetricLogger, SmoothedValue, form_stats
from computer_vision.video_mae.optim_factory import LayerDecayValueAssigner, create_optimizer
from computer_vision.video_mae.engine_for_finetuning import  accuracy

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
data_dirpath=Path('D:/data/UCF101')
root=data_dirpath/'UCF-101'
train_annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask/trainlist01.txt'
val_annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask/vallist01.txt'

mini_train=False
if not mini_train:
    pretrain_path=Path('D:/results/ucf101/video_mae/train/checkpoints/last.pth') 
    output_dirpath=Path('D:/results/ucf101/video_mae/finetune') 
    arguments= f"""--data_root {root} --train_data_path {train_annotation_path} --val_data_path {val_annotation_path} 
    --output_dir {output_dirpath} --finetune {pretrain_path} 
    --data_set UCF101 --nb_classes 101  --batch_size 3 --input_size 224 --cutmix 1. --mixup 0.8
    --short_side_size 224 --num_frames 16 --sampling_rate 4 --num_sample 2 --num_workers 0 --opt adamw
    --cos_attn --lr 1e-3 --drop_path 0.3 --clip_grad 5.0 --layer_decay 0.9 --opt_betas 0.9 0.999 --weight_decay 0.1
    --test_num_segment 5 --test_num_crop 3 --dist_eval
    --warmup_epochs 5 --epochs 35  --print_freq 20 --device cuda --time 12 --resume
    """ # --use-cutmix-mixup
else:
    pretrain_path=Path('D:/results/ucf101/video_mae/mini_train/checkpoints/last.pth') 
    output_dirpath=Path('D:/results/ucf101/video_mae/mini_finetune') 
    arguments= f"""--data_root {root} --train_data_path {train_annotation_path} --val_data_path {val_annotation_path} 
    --output_dir {output_dirpath}  --finetune {pretrain_path} 
    --data_set UCF101 --nb_classes 101  --batch_size 3 --input_size 224 --cutmix 1. --mixup 0.8
    --short_side_size 224 --num_frames 16 --sampling_rate 4 --num_sample 2 --num_workers 0 --opt adamw
    --cos_attn --lr 1e-3 --drop_path 0.3 --clip_grad 5.0 --layer_decay 0.9 --opt_betas 0.9 0.999 --weight_decay 0.1
    --test_num_segment 5 --test_num_crop 3 --dist_eval 
    --warmup_epochs 5 --print_freq 20 --epochs 35  --device cuda --resume
    --n_steps 3 --n_epochs 4 --time 0.5
    """ 

known_args, _=parser.parse_known_args(args=arguments.split())
if known_args.enable_deepspeed:
    parser=deepspeed.add_config_arguments(parser)
    ds_init=deepspeed.initialize
else: ds_init=None
args=parser.parse_args(arguments.split())


In [11]:
args.output_dir=Path(args.output_dir)
args.checkpoint_dir=args.output_dir/"checkpoints"
assert args.output_dir.is_dir(),f'{args.output_dir} does not exist'
assert args.checkpoint_dir.is_dir(),f'{args.checkpoint_dir} does not exist'
args.last=args.checkpoint_dir/args.last
args.best=args.checkpoint_dir/args.best
assert args.last.is_file(), f"{args.last} does not exist"
assert args.best.is_file(), f"{args.last} does not exist"
print(f"{args.last=}")
print(f"{args.best=}")

device=torch.device(args.device) if (torch.cuda.is_available() and args.device=='cuda') else torch.device('cpu')
print(f"{device=}")


args.last=WindowsPath('D:/results/ucf101/video_mae/finetune/checkpoints/last.pth')
args.best=WindowsPath('D:/results/ucf101/video_mae/finetune/checkpoints/best.pth')
device=device(type='cuda')


In [12]:
args.nb_classes=101
dataset_test=VideoClsDataset(data_root=args.data_root, anno_path=args.val_data_path, mode='test', clip_len=args.num_frames, 
                        frame_sample_rate=args.sampling_rate, num_segment=1, test_num_segment=args.test_num_segment,
                        test_num_crop=args.test_num_crop, num_crop=3, keep_aspect_ratio=True, crop_size=args.input_size,
                        short_side_size=args.short_side_size, new_height=256, new_width=320, args=args)

if args.num_sample>1: collate_func=partial(multiple_samples_collate, fold=False)
num_devices=torch.cuda.device_count() # number of CUDA devices
data_loader_test=torch.utils.data.DataLoader(dataset_test, batch_size=args.batch_size, num_workers=args.num_workers, shuffle=False,
                                             pin_memory=num_devices>0 and args.pin_mem, drop_last=False, persistent_workers=args.num_workers>0, 
                                             worker_init_fn=seed_worker)


In [13]:
model=vit_tiny_patch16_224(pretrained=False, img_size=args.input_size, num_classes=args.nb_classes, all_frames=args.num_frames*args.num_segments,
                          tubelet_size=args.tubelet_size, drop_rate=args.drop, drop_path_rate=args.drop_path, attn_drop_rate=args.attn_drop_rate,
                          head_drop_rate=args.head_drop_rate, use_mean_pooling=args.use_mean_pooling, init_scale=args.init_scale,
                          with_cp=args.with_checkpoint, cos_attn=args.cos_attn)
print(f"{args.tubelet_size=}, {args.drop=}, {args.drop_path=}, {args.attn_drop_rate=}, {args.head_drop_rate=}., {args.use_mean_pooling=}")
print(f"{args.init_scale=}, {args.with_checkpoint=}")
args.patch_size=model.patch_embed.patch_size
args.window_size=(args.num_frames//args.tubelet_size, args.input_size//args.patch_size[0], args.input_size//args.patch_size[1])
print(f"Patch size: {args.patch_size}, Number of patches per dim: {args.window_size}")

checkpoint=None
if args.last.is_file():
    checkpoint=torch.load(args.last, map_location='cpu', weights_only=False)
    print(f"Resume from checkpoint: {args.last}")
    model.load_state_dict(checkpoint['model'])
assert checkpoint is not None, f"Failed to load checkpoint from {args.last}"

model.to(device)
model.eval()
n_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model contains {n_parameters} parameters trained for {checkpoint['epoch']} epochs")

args.tubelet_size=2, args.drop=0.0, args.drop_path=0.3, args.attn_drop_rate=0.0, args.head_drop_rate=0.0., args.use_mean_pooling=True
args.init_scale=0.001, args.with_checkpoint=False
Patch size: (16, 16), Number of patches per dim: (8, 14, 14)
Resume from checkpoint: D:\results\ucf101\video_mae\finetune\checkpoints\last.pth
Model contains 4768649 parameters trained for 49 epochs


In [14]:
criterion=torch.nn.CrossEntropyLoss()

metric_logger=MetricLogger(delimiter=" ")
header='Test: '

for idx, batch in enumerate(metric_logger.log_every(data_loader_test, args.print_freq, header)):

    if args.n_steps is not None and idx > args.n_steps-1:
        print(f"Hit desired number of steps: {idx}/{args.n_steps}")
        break
    images, target, fname, chunk_nb, split_nb=batch
    images=images.to(device=device, non_blocking=device==torch.device('cuda'))
    target=target.to(device=device, non_blocking=device==torch.device('cuda'))

    with torch.no_grad():
        output=model(images)
        loss=criterion(output, target)

    acc1, acc5=accuracy(output, target, topk=(1,5))
    metric_logger.update(loss=loss.item())
    metric_logger.meters['acc1'].update(acc1.item(), n=args.batch_size)
    metric_logger.meters['acc5'].update(acc1.item(), n=args.batch_size)
    
print(f"Acc@1 {metric_logger.acc1.global_avg:.3f} Acc@5 {metric_logger.acc5.global_avg:.3f} loss {metric_logger.loss.global_avg:.3f}")

Test:  [    0/18915] eta:2:47:48 loss:2.5102 (2.5102) acc1:66.6667 (66.6667) acc5:66.6667 (66.6667) time: 0.5323 (0.5323 -- 0.5323) data: 0.1459 (0.1459 -- 0.1459) max mem: 241
Test:  [   20/18915] eta:1:05:41 loss:2.1547 (2.2244) acc1:100.0000 (65.0794) acc5:100.0000 (65.0794) time: 0.1924 (0.1447 -- 0.2916) data: 0.1506 (0.1132 -- 0.2361) max mem: 241
Test:  [   40/18915] eta:1:08:21 loss:3.5062 (2.8622) acc1:0.0000 (40.6504) acc5:0.0000 (40.6504) time: 0.2264 (0.1506 -- 0.3940) data: 0.1565 (0.0761 -- 0.3183) max mem: 241
Test:  [   60/18915] eta:1:08:01 loss:3.6264 (3.1636) acc1:0.0000 (28.9617) acc5:0.0000 (28.9617) time: 0.2147 (0.1675 -- 0.3033) data: 0.1463 (0.0970 -- 0.2388) max mem: 241
Test:  [   80/18915] eta:1:05:51 loss:2.3969 (2.9923) acc1:33.3333 (34.5679) acc5:33.3333 (34.5679) time: 0.1895 (0.1325 -- 0.3346) data: 0.1208 (0.0683 -- 0.2543) max mem: 241
Test:  [  100/18915] eta:1:02:03 loss:2.9960 (2.9759) acc1:0.0000 (34.6535) acc5:0.0000 (34.6535) time: 0.1497 (0.113